# DAMPS-MMHCL Training for Amazon Clothing Dataset (Local)

This notebook trains the **DAMPS-MMHCL** (Spectral Domain Representation Calibration upgrade for the Multi-Modal Hypergraph Contrastive Learning framework) on the Amazon Clothing dataset on a local machine with NVIDIA RTX 5090 (32 GB VRAM).

It is a drop-in replacement for the original MMHCL notebook and follows the *Revision 9 — 100% Compliance Check & Final Lock* architecture spec (see `DAMPS_to_MMHCL_architecture_revision42.tex` / `.pdf`).

## DAMPS pipeline (per forward pass)
1. **Spectral decomposition** — 1-D rFFT on the projected modality features (d = 64).
2. **Metadata-Aware APC** — von-Mises MLE phase calibration grouped by static metadata categories (no k-means).
3. **AVRF** — logit-clipped Wiener gate with strict `[-2, 2]` initialisation prior (data-driven from raw modality SNR) and per-epoch EMA MAD aggregation.
4. **Residual IMCF** — ASC consensus coefficient applied in residual form to avoid multiplicative attenuation.
5. **Inverse FFT** — back to spatial domain → fed into HGCN via **Soft Residual-Routing** (eq. 3 of the spec).

Auxiliary upgrades wired into the trainer:
- **Pattern B' (Scheduled Rebuild)** — reconstruct the K-NN multi-modal hypergraph every `R` epochs from the **Slim Momentum** buffers (98 % VRAM saving vs MoCo).
- **Dual-path K-NN** — chunked PyTorch (default) with FAISS HNSW fallback when N ≥ 60 000.
- **`bfloat16` mixed precision** (-30 / -40 % wall-clock, -40 % VRAM on Ada Lovelace).
- **Learnable InfoNCE temperature τ** (Section 3.1).
- **cuFFT plan-cache lockdown** (`cufft_plan_cache.max_size = 1`, Section 3.3).
- **Diagnostic logs** — NNZ / avg_deg per rebuild, tanh-saturation rate of the AVRF gates.

## Configuration
- **GPU**: NVIDIA RTX 5090 (32 GB VRAM)
- **Working dir**: `MMHCL_DAMPS_Project/`
- **Entry point**: `train.py` (replaces the original `codes/main.py`)
- **Batch Size**: 1024
- **Max Epochs**: 250
- **Rebuild cadence**: every 5 epochs after a 10-epoch warm-up
- **W&B Project**: `damps-mmhcl-clothing`

## Steps
1. Verify environment and GPU
2. Install dependencies for the DAMPS-MMHCL package
3. Set up W&B logging
4. (Optional) GPU memory monitor
5. Run a quick smoke test of the DAMPS package
6. Multi-seed training of the full DAMPS-MMHCL model
7. Aggregate results & export to Excel

## 1. Environment Setup & GPU Verification

In [1]:
from __future__ import annotations

import os
import sys
import torch

# ---------------------------------------------------------------------------
#  Resolve project root.  The notebook may be opened from either the
#  workspace root or one of the package sub-directories.
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(('codes', 'MMHCL_DAMPS_Project')):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

# DAMPS-MMHCL implementation lives under MMHCL_DAMPS_Project/
DAMPS_DIR: str = os.path.join(PROJECT_ROOT, 'MMHCL_DAMPS_Project')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')
# Legacy CODES_DIR is still around for the original MMHCL baseline ablation
CODES_DIR: str = os.path.join(PROJECT_ROOT, 'codes')

print(f"Project Root        : {PROJECT_ROOT}")
print(f"DAMPS-MMHCL Directory: {DAMPS_DIR}")
print(f"Data Directory       : {DATA_DIR}")

assert os.path.exists(DAMPS_DIR), (
    f"MMHCL_DAMPS_Project not found at {DAMPS_DIR}. "
    f"Make sure you cloned the full DAMPS-MMHCL repository."
)
assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
print("\n[OK] All directories verified")

# Verify GPU & bfloat16 capability (DAMPS uses bf16 AMP by default)
print("\n" + "=" * 60)
print("GPU Information")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    total_gb: float = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU Memory  : {total_gb:.1f} GB")

    # bfloat16 is a hard requirement for the DAMPS bf16 AMP path. Detect
    # support so the user gets a clear error early instead of a cryptic
    # autocast failure mid-training.
    bf16_ok: bool = torch.cuda.is_bf16_supported()
    print(f"bfloat16 support: {bf16_ok}")
    if not bf16_ok:
        print(
            "[WARNING] bfloat16 not supported on this GPU. Pass --use_amp 0 "
            "to fall back to fp32, or run on Ampere/Ada Lovelace+."
        )
else:
    print("[WARNING] No GPU detected. DAMPS training will be very slow on CPU.")

Project Root        : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing
DAMPS-MMHCL Directory: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
Data Directory       : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data

[OK] All directories verified

GPU Information
CUDA available: True
GPU         : NVIDIA GeForce RTX 5090
CUDA version: 12.8
GPU Memory  : 31.8 GB
bfloat16 support: True


## 1.5 Install Dependencies (Run Once)

In [2]:
# Install required dependencies for the DAMPS-MMHCL package (run once).
#
# The DAMPS-MMHCL implementation pins its requirements in
# `MMHCL_DAMPS_Project/requirements.txt`. We install from there first,
# then add notebook-only extras (wandb, pandas, openpyxl).
from __future__ import annotations

import subprocess
import sys
import os

print("=" * 60)
print("Installing DAMPS-MMHCL dependencies")
print("=" * 60)

# UV-managed Python sometimes refuses installs without --break-system-packages
pip_extra_args: list[str] = []
test_result: subprocess.CompletedProcess[str] = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', 'pip'],
    capture_output=True, text=True,
)
if 'externally-managed' in test_result.stderr:
    pip_extra_args = ['--break-system-packages']
    print("\n[INFO] UV-managed Python detected, using --break-system-packages")

# Step 1 - install DAMPS-MMHCL package requirements
requirements_path: str = os.path.join(DAMPS_DIR, 'requirements.txt')
if not os.path.exists(requirements_path):
    requirements_path = os.path.join(PROJECT_ROOT, 'requirements.txt')

if os.path.exists(requirements_path):
    print(f"\n1. Installing packages from: {requirements_path}")
    print("   This may take several minutes for PyTorch (~2.6 GB)...")
    cmd: list[str] = [sys.executable, '-m', 'pip', 'install', '-r', requirements_path] + pip_extra_args
    result: subprocess.CompletedProcess[str] = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"   [ERROR] Installation failed! Exit code: {result.returncode}")
        print("\n--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise Exception("Failed to install requirements")
    print("   [OK] DAMPS-MMHCL package requirements installed")
else:
    print(f"   [WARNING] No requirements.txt found at {requirements_path}")

# Step 2 - install notebook-only extras
additional_packages: list[str] = ['wandb', 'openpyxl', 'pandas']
print(f"\n2. Installing notebook extras: {additional_packages}")
for package in additional_packages:
    print(f"   Installing {package}...")
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', package] + pip_extra_args
    subprocess.run(cmd, capture_output=True)
print("   [OK] Notebook extras installed")

# Step 3 - sanity-check imports of all DAMPS-MMHCL modules
print("\n3. Verifying DAMPS-MMHCL package imports...")
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)
try:
    import damps  # noqa: F401
    from damps import (  # noqa: F401
        DAMPS,
        SlimMomentumEncoder,
        DualPathKNN,
        compute_avrf_prior,
        compute_avrf_logit,
    )
    print("   [OK] `damps` core package imports cleanly")
except Exception as exc:  # pragma: no cover
    print(f"   [WARNING] Could not import `damps` package yet: {exc}")
    print("   This is expected on the very first run; restart the kernel and try again.")

print("\n" + "=" * 60)
print("[OK] All packages installed successfully!")
print("=" * 60)
print("\n[IMPORTANT] If this is the first install, RESTART THE KERNEL before running the next cells.")

Installing DAMPS-MMHCL dependencies

1. Installing packages from: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\requirements.txt
   This may take several minutes for PyTorch (~2.6 GB)...
   [OK] DAMPS-MMHCL package requirements installed

2. Installing notebook extras: ['wandb', 'openpyxl', 'pandas']
   Installing wandb...
   Installing openpyxl...
   Installing pandas...
   [OK] Notebook extras installed

3. Verifying DAMPS-MMHCL package imports...
   [OK] `damps` core package imports cleanly

[OK] All packages installed successfully!

[IMPORTANT] If this is the first install, RESTART THE KERNEL before running the next cells.


## 2. Weights & Biases Setup

In [ ]:
from __future__ import annotations

import wandb
from IPython.display import display, Markdown

# ========== Login to Weights & Biases ==========
# A dedicated project so DAMPS runs do not get mixed in with the original
# MMHCL baseline runs. Change `wandb_entity` if you push to your own account.
wandb_project: str = 'damps-mmhcl-clothing'
wandb_entity: str = 'baitapck51cc-uet'

# Your W&B API key (replace with your own if you fork this notebook)
WANDB_API_KEY: str = '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'

wandb.login(key=WANDB_API_KEY)

print("[OK] W&B login successful")
print(f"     Entity : {wandb_entity}")
print(f"     Project: {wandb_project}")

wandb_url: str = f"https://wandb.ai/{wandb_entity}/{wandb_project}"
print()
display(Markdown(f"## [Open Weights & Biases Dashboard]({wandb_url})"))
print(f"Direct Link: {wandb_url}")

## 2.5 GPU Memory Monitor (Optional)

Start a lightweight background monitor to print GPU VRAM usage while training.
Run the stop function after training completes.

In [ ]:
from __future__ import annotations

import threading
import time
import torch
import csv
import os
from typing import Optional

# GPU monitor controls
GPU_MONITOR_RUNNING: bool = False
GPU_MONITOR_THREAD: Optional[threading.Thread] = None


def _resolve_monitor_root() -> str:
    current_dir: str = os.getcwd()
    if current_dir.endswith('codes'):
        return os.path.dirname(current_dir)
    return current_dir


def _gpu_monitor(interval_seconds: int, csv_path: str, print_live: bool) -> None:
    global GPU_MONITOR_RUNNING
    if not torch.cuda.is_available():
        print("[WARNING] CUDA not available. GPU monitor will not run.")
        GPU_MONITOR_RUNNING = False
        return

    total_mem: int = torch.cuda.get_device_properties(0).total_memory
    start_time: float = time.time()

    file_exists: bool = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as csv_file:
        writer: csv.writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow([
                'relative_time_s',
                'allocated_gb',
                'reserved_gb',
                'allocated_pct',
                'reserved_pct'
            ])

        print(f"[INFO] GPU monitor started. Logging to: {csv_path}")
        while GPU_MONITOR_RUNNING:
            allocated: int = torch.cuda.memory_allocated(0)
            reserved: int = torch.cuda.memory_reserved(0)
            allocated_pct: float = (allocated / total_mem) * 100
            reserved_pct: float = (reserved / total_mem) * 100
            rel_time: float = time.time() - start_time

            writer.writerow([
                f"{rel_time:.2f}",
                f"{allocated / 1024**3:.4f}",
                f"{reserved / 1024**3:.4f}",
                f"{allocated_pct:.2f}",
                f"{reserved_pct:.2f}",
            ])
            csv_file.flush()

            if print_live:
                print(
                    f"[GPU] allocated={allocated/1024**3:.2f} GB ({allocated_pct:.1f}%), "
                    f"reserved={reserved/1024**3:.2f} GB ({reserved_pct:.1f}%)"
                )

            time.sleep(interval_seconds)

    print("[INFO] GPU monitor stopped.")


def start_gpu_monitor(interval_seconds: int = 10, csv_filename: str = 'gpu_memory_monitor.csv', print_live: bool = False) -> None:
    """Start background GPU monitor. Logs to CSV every N seconds."""
    global GPU_MONITOR_RUNNING, GPU_MONITOR_THREAD
    if GPU_MONITOR_RUNNING:
        print("[INFO] GPU monitor already running.")
        return

    monitor_root: str = _resolve_monitor_root()
    csv_path: str = os.path.join(monitor_root, csv_filename)

    GPU_MONITOR_RUNNING = True
    GPU_MONITOR_THREAD = threading.Thread(
        target=_gpu_monitor,
        args=(interval_seconds, csv_path, print_live),
        daemon=True
    )
    GPU_MONITOR_THREAD.start()


def stop_gpu_monitor() -> None:
    """Stop background GPU monitor."""
    global GPU_MONITOR_RUNNING
    GPU_MONITOR_RUNNING = False


# Start monitor with 15s interval (adjust as needed)
# Set print_live=True if you want console output too
start_gpu_monitor(interval_seconds=15, csv_filename='gpu_memory_monitor.csv', print_live=False)

# Call stop_gpu_monitor() after training completes.

## 2.6 DAMPS Smoke Test (Optional)

Run a small end-to-end sanity check of the DAMPS-MMHCL package before launching the full multi-seed training. This invokes `MMHCL_DAMPS_Project/tests/smoke_test.py`, which exercises:

- AVRF prior derivation + logit clipping
- DAMPS core forward pass (FFT → APC → AVRF → Residual IMCF → iFFT)
- Slim Momentum Encoder update step
- Dual-Path KNN graph construction
- Full `DAMPS_MMHCL` model forward + backward

Skip this cell if you have already validated the package.

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess

# DAMPS_DIR was defined in the environment-setup cell.
smoke_path: str = os.path.join(DAMPS_DIR, 'tests', 'smoke_test.py')

if not os.path.exists(smoke_path):
    print(f"[WARN] Smoke test not found at {smoke_path} - skipping.")
else:
    print("=" * 70)
    print("Running DAMPS-MMHCL smoke test...")
    print(f"  Script: {smoke_path}")
    print("=" * 70)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    env['PYTHONUNBUFFERED'] = '1'

    result = subprocess.run(
        [sys.executable, smoke_path],
        cwd=DAMPS_DIR,
        capture_output=True,
        text=True,
        env=env,
        encoding='utf-8',
        errors='replace',
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("\n[FAIL] Smoke test exited with code", result.returncode)
        if result.stderr:
            print("\n--- STDERR ---")
            print(result.stderr)
    else:
        print("\n[OK] DAMPS-MMHCL smoke test passed.")

## 3. Multi-Seed Training (DAMPS-MMHCL)

Train the **DAMPS-MMHCL** model with multiple random seeds to obtain mean ± std and (downstream) confidence intervals + paired t-tests. This calls `MMHCL_DAMPS_Project/train.py` directly — the same entry-point used by `scripts/run_clothing.bat`.

### Backbone hyperparameters (parity with original MMHCL paper)
- **Batch Size**: 1024 (RTX 5090, 32 GB VRAM)
- **Max Epochs**: 250
- **Learning Rate**: 0.0001
- **Embedding Dim**: 64 / **Top-k**: 5 / **τ₀**: 0.1 (learnable; spec Section 3.1)
- **`User_layers` / `Item_layers`**: 3 / 2
- **`user_loss_ratio` / `item_loss_ratio`**: 0.03 / 0.07
- **Early Stopping**: patience = 30, min_epochs = 75, monitor = `val_recall@20`
- **ReduceLROnPlateau**: factor = 0.5, patience = 3

### DAMPS-specific flags (Revision 9)
- `--damps_apc 1` — Metadata-Aware Adaptive Phase Calibration
- `--damps_avrf 1` — Anti-Variance Resilient Filter (logit-clipped Wiener gate)
- `--damps_imcf 1` — Residual Inter-Modal Coherence Filter
- `--damps_soft_routing 1` — Soft Residual-Routing into HGCN
- `--damps_momentum 1` — Slim Momentum Encoder
- `--damps_data_driven_prior 1` — SNR-derived AVRF priors
- `--damps_num_categories 10` — # static metadata clusters for APC
- `--damps_warmup_epochs 10` — adaptive EMA warm-up
- `--rebuild_R 5` — Pattern B' rebuild cadence
- `--knn_chunk_size 4096` / `--faiss_threshold 60000` — Dual-path K-NN
- `--use_amp 1` — bfloat16 mixed precision
- `--clip_grad_norm 1.0` — gradient clipping

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
import json
import re
import numpy as np
import time
import random
from pathlib import Path
from typing import Any, Optional

# ---------------------------------------------------------------------------
#  Resolve project directories. The DAMPS-MMHCL package lives under
#  MMHCL_DAMPS_Project/. ``train.py`` writes its logs relative to its own
#  cwd as ``../<dataset>/<path_name>/<path_name>.txt``, so we must launch
#  the subprocess with cwd = MMHCL_DAMPS_Project.
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(('codes', 'MMHCL_DAMPS_Project')):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

DAMPS_DIR: str = os.path.join(PROJECT_ROOT, 'MMHCL_DAMPS_Project')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')

# Switch into the DAMPS package directory so relative paths in train.py
# (``../{dataset}/...``) resolve to the workspace root.
if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
    os.chdir(DAMPS_DIR)
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)

PYTHON_EXE: str = sys.executable

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Working directory : {os.getcwd()}")
print(f"Python executable : {PYTHON_EXE}")

print("\n" + "=" * 80)
print("MULTI-SEED TRAINING: Running 3 DAMPS-MMHCL experiments")
print("=" * 80)

# Generate reproducible-but-fresh seeds based on current time
base_seed: int = int(time.time_ns() % (2 ** 31))
random.seed(base_seed)

n_runs: int = 3
seeds: list[int] = [random.randint(1, 2 ** 31 - 1) for _ in range(n_runs)]

print(f"\nGenerated {n_runs} random seeds based on current time:")
print(f"Base timestamp seed : {base_seed}")
print(f"Seeds for experiments: {seeds}")
print(f"(These seeds are saved in the summary file for reproducibility)")

# W&B configuration (already created at login time, kept in sync here)
wandb_project: str = 'damps-mmhcl-clothing'
wandb_entity: str = 'baitapck51cc-uet'

# ---------------------------------------------------------------------------
#  Backbone hyper-parameters (parity with the original MMHCL paper)
# ---------------------------------------------------------------------------
dataset: str = 'Clothing'
gpu_id: int = 0
epoch: int = 250
verbose: int = 5

batch_size: int = 1024
lr: float = 0.0001
regs: float = 1e-3
embed_size: int = 64
topk: int = 5
core: int = 5
User_layers: int = 3
Item_layers: int = 2
user_loss_ratio: float = 0.03
item_loss_ratio: float = 0.07
temperature: float = 0.1  # initialisation of the *learnable* tau (Revision 9 spec, Section 3.1)
clip_grad_norm: float = 1.0

# Early stopping
early_stopping_patience: int = 30
early_stopping_min_epochs: int = 75
early_stopping_min_delta: float = 0.0001
early_stopping_mode: str = 'max'
early_stopping_restore_best: int = 1

# ReduceLROnPlateau
use_reduce_lr: int = 1
reduce_lr_factor: float = 0.5
reduce_lr_patience: int = 3
reduce_lr_min: float = 1e-6

# ---------------------------------------------------------------------------
#  DAMPS-specific configuration (Revision 9, full feature set)
# ---------------------------------------------------------------------------
damps_apc: int = 1                    # Metadata-Aware Adaptive Phase Calibration
damps_avrf: int = 1                   # Logit-clipped Wiener gate
damps_imcf: int = 1                   # Residual Inter-Modal Coherence Filter
damps_permutation_fft: int = 0        # Falsifiability ablation (off in main runs)
damps_soft_routing: int = 1           # h_input = h_raw + alpha * LN(h_cal)
damps_momentum: int = 1               # Slim Momentum Encoder
damps_data_driven_prior: int = 1      # SNR-derived AVRF priors
damps_num_categories: int = 10        # Static metadata clusters for APC
damps_warmup_epochs: int = 10         # Adaptive EMA warm-up
rebuild_R: int = 5                    # Pattern B' cadence
faiss_threshold: int = 60_000         # FAISS HNSW fallback threshold
knn_chunk_size: int = 4096            # Chunked PyTorch K-NN tile size
use_amp: int = 1                      # bfloat16 mixed precision

# Storage for results
all_results: list[dict[str, Any]] = []

print(f"\n{'=' * 80}")
print("Training Configuration")
print(f"{'=' * 80}")
print(f"  Dataset            : {dataset}")
print(f"  GPU ID             : {gpu_id}")
print(f"  Max Epochs         : {epoch}")
print(f"  Batch Size         : {batch_size}")
print(f"  Learning Rate      : {lr}")
print(f"  Early Stopping     : patience={early_stopping_patience}, min_epochs={early_stopping_min_epochs}")
print(f"  Mixed Precision    : bfloat16  (use_amp={use_amp})")
print(f"  Rebuild Cadence    : every {rebuild_R} epochs (Pattern B')")
print(f"  DAMPS Switches     : APC={damps_apc} | AVRF={damps_avrf} | IMCF={damps_imcf} | "
      f"Soft-Routing={damps_soft_routing} | Momentum={damps_momentum}")
print(f"  W&B Project        : {wandb_project}")


def _build_path_name(seed: int) -> str:
    """Mirror the directory naming scheme of ``train.py::_experiment_paths``."""
    return (
        f"damps_uu_ii={User_layers}_{Item_layers}"
        f"_{user_loss_ratio}_{item_loss_ratio}"
        f"_topk={topk}_t={temperature}_R={rebuild_R}"
        f"_apc={damps_apc}_avrf={damps_avrf}_imcf={damps_imcf}"
        f"_regs={regs}_dim={embed_size}_seed={seed}_"
    )


# Run training for each seed
for run_idx, seed in enumerate(seeds, 1):
    print(f"\n{'=' * 80}")
    print(f"RUN {run_idx}/{n_runs}: Training DAMPS-MMHCL with seed={seed}")
    print(f"{'=' * 80}")

    cmd: list[str] = [
        PYTHON_EXE, 'train.py',
        # Data / schedule
        '--dataset', dataset,
        '--gpu_id', str(gpu_id),
        '--seed', str(seed),
        '--epoch', str(epoch),
        '--verbose', str(verbose),
        '--batch_size', str(batch_size),
        '--lr', str(lr),
        '--regs', str(regs),
        '--clip_grad_norm', str(clip_grad_norm),
        # Model architecture
        '--embed_size', str(embed_size),
        '--topk', str(topk),
        '--core', str(core),
        '--User_layers', str(User_layers),
        '--Item_layers', str(Item_layers),
        '--user_loss_ratio', str(user_loss_ratio),
        '--item_loss_ratio', str(item_loss_ratio),
        '--temperature', str(temperature),
        # Early stopping
        '--early_stopping_patience', str(early_stopping_patience),
        '--early_stopping_min_epochs', str(early_stopping_min_epochs),
        '--early_stopping_min_delta', str(early_stopping_min_delta),
        '--early_stopping_mode', str(early_stopping_mode),
        '--early_stopping_restore_best', str(early_stopping_restore_best),
        # ReduceLROnPlateau
        '--use_reduce_lr', str(use_reduce_lr),
        '--reduce_lr_factor', str(reduce_lr_factor),
        '--reduce_lr_patience', str(reduce_lr_patience),
        '--reduce_lr_min', str(reduce_lr_min),
        # DAMPS-specific
        '--damps_apc', str(damps_apc),
        '--damps_avrf', str(damps_avrf),
        '--damps_imcf', str(damps_imcf),
        '--damps_permutation_fft', str(damps_permutation_fft),
        '--damps_soft_routing', str(damps_soft_routing),
        '--damps_momentum', str(damps_momentum),
        '--damps_data_driven_prior', str(damps_data_driven_prior),
        '--damps_num_categories', str(damps_num_categories),
        '--damps_warmup_epochs', str(damps_warmup_epochs),
        # Pattern B' rebuild
        '--rebuild_R', str(rebuild_R),
        '--faiss_threshold', str(faiss_threshold),
        '--knn_chunk_size', str(knn_chunk_size),
        # Mixed precision
        '--use_amp', str(use_amp),
        # W&B
        '--use_wandb', '1',
        '--wandb_project', wandb_project,
        '--wandb_entity', wandb_entity,
        '--wandb_run_name', f'seed_{seed}',
    ]

    print(f"Command: {' '.join(cmd)}")
    print(f"Current directory: {os.getcwd()}\n")

    env: dict[str, str] = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    env['PYTHONUNBUFFERED'] = '1'

    result: subprocess.CompletedProcess[str] = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=env,
        encoding='utf-8',
        errors='replace',
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        print(f"\n[WARNING] DAMPS run with seed={seed} exited with code {result.returncode}")
        if result.stderr:
            print("\n[ERROR OUTPUT]:")
            print(result.stderr)
    else:
        print(f"\n[OK] DAMPS run with seed={seed} completed successfully")

    # ----- Extract metrics from the per-run log file ---------------------
    # train.py writes ``../{dataset}/{path_name}/{path_name}.txt`` relative
    # to MMHCL_DAMPS_Project/, which is also the cwd inside this notebook.
    path_name: str = _build_path_name(seed)
    log_file: str = f"../{dataset}/{path_name}/{path_name}.txt"

    if os.path.exists(log_file):
        with open(log_file, 'r', encoding='utf-8') as f:
            log_content: str = f.read()

        # Accept either ":" or "=" between key and value to remain
        # compatible with intermediate logger formats.
        sep: str = r"[:=]\s*"
        recall_match: Optional[re.Match[str]] = (
            re.search(rf'BEST_Test_Recall@20{sep}([\d.]+)', log_content)
            or re.search(rf'Test_Recall@20{sep}([\d.]+)', log_content)
        )
        precision_match: Optional[re.Match[str]] = (
            re.search(rf'BEST_Test_Precision@20{sep}([\d.]+)', log_content)
            or re.search(rf'Test_Precision@20{sep}([\d.]+)', log_content)
        )
        ndcg_match: Optional[re.Match[str]] = (
            re.search(rf'BEST_Test_NDCG@20{sep}([\d.]+)', log_content)
            or re.search(rf'Test_NDCG@20{sep}([\d.]+)', log_content)
        )

        # ----- Optional DAMPS diagnostics --------------------------------
        # train.py logs lines such as:
        #   "[diag] tau=0.612  alpha=[0.97 0.95 ...]  tanh_sat=0.31"
        #   "[rebuild] epoch=10  nnz=458_321  avg_deg=19.86"
        tau_match: Optional[re.Match[str]] = re.search(r'tau\s*=\s*([\d.]+)', log_content)
        sat_match: Optional[re.Match[str]] = re.search(r'tanh_sat\s*=\s*([\d.]+)', log_content)
        nnz_match: Optional[re.Match[str]] = re.search(r'nnz\s*=\s*([\d_]+)', log_content)
        avg_deg_match: Optional[re.Match[str]] = re.search(r'avg_deg\s*=\s*([\d.]+)', log_content)

        if recall_match and ndcg_match:
            precision_value: Optional[float] = (
                float(precision_match.group(1)) if precision_match is not None else None
            )
            run_result: dict[str, Any] = {
                'seed': seed,
                'recall@20': float(recall_match.group(1)),
                'precision@20': precision_value,
                'ndcg@20': float(ndcg_match.group(1)),
                'tau_final': float(tau_match.group(1)) if tau_match else None,
                'tanh_sat_final': float(sat_match.group(1)) if sat_match else None,
                'last_nnz': int(nnz_match.group(1).replace('_', '')) if nnz_match else None,
                'last_avg_deg': float(avg_deg_match.group(1)) if avg_deg_match else None,
                'log_file': log_file,
            }
            all_results.append(run_result)
            precision_str: str = (
                f"{precision_value:.6f}" if precision_value is not None else "n/a"
            )
            print(
                f"  Extracted metrics: "
                f"Recall@20={run_result['recall@20']:.6f}, "
                f"Precision@20={precision_str}, "
                f"NDCG@20={run_result['ndcg@20']:.6f}"
            )
            if run_result['tau_final'] is not None:
                print(f"  DAMPS diagnostics : tau={run_result['tau_final']:.4f}, "
                      f"tanh_sat={run_result['tanh_sat_final']}, "
                      f"last_nnz={run_result['last_nnz']}, "
                      f"last_avg_deg={run_result['last_avg_deg']}")
        else:
            print(f"  [WARN] Could not extract metrics from log file: {log_file}")
    else:
        print(f"  [WARN] Log file not found: {log_file}")

print(f"\n{'=' * 80}")
print("ALL DAMPS-MMHCL TRAINING RUNS COMPLETED")
print(f"{'=' * 80}")

## 4. Results Summary & Statistics

In [ ]:
from __future__ import annotations

from typing import Any
import numpy as np
import json

# ---------------------------------------------------------------------------
#  Aggregate DAMPS-MMHCL multi-seed results.
#
#  ``all_results`` is populated by the previous cell. Each entry has at
#  minimum: seed, recall@20, ndcg@20 (precision@20 is optional in case it
#  is missing from very old log files). DAMPS diagnostics (tau_final,
#  tanh_sat_final, last_nnz, last_avg_deg) are also reported when present.
# ---------------------------------------------------------------------------
if len(all_results) > 0:
    print(f"\n{'=' * 80}")
    print(f"DAMPS-MMHCL RESULTS SUMMARY (Mean ± Std across {len(all_results)} runs)")
    print(f"{'=' * 80}")

    metrics: list[str] = ['recall@20', 'precision@20', 'ndcg@20']
    stats: dict[str, dict[str, Any]] = {}

    for metric in metrics:
        raw_values: list[Any] = [r.get(metric) for r in all_results]
        values: list[float] = [float(v) for v in raw_values if v is not None]

        if not values:
            stats[metric] = {'mean': float('nan'), 'std': float('nan'), 'values': []}
            print(f"\n{metric.upper()}: no values available")
            continue

        mean_val: np.floating = np.mean(values)
        std_val: np.floating = np.std(values)
        stats[metric] = {'mean': float(mean_val), 'std': float(std_val), 'values': values}

        print(f"\n{metric.upper()}:")
        print(f"  Mean: {mean_val:.6f}")
        print(f"  Std:  {std_val:.6f}")
        print(f"  Mean ± Std: {mean_val:.6f} ± {std_val:.6f}")
        print(f"  Individual values: {[f'{v:.6f}' for v in values]}")

    # ---- DAMPS diagnostic aggregates --------------------------------------
    diag_keys: list[str] = ['tau_final', 'tanh_sat_final', 'last_nnz', 'last_avg_deg']
    diag_summary: dict[str, dict[str, Any]] = {}
    for k in diag_keys:
        raw: list[Any] = [r.get(k) for r in all_results]
        clean: list[float] = [float(v) for v in raw if v is not None]
        if clean:
            diag_summary[k] = {
                'mean': float(np.mean(clean)),
                'std': float(np.std(clean)),
                'values': clean,
            }

    if diag_summary:
        print(f"\n{'=' * 80}")
        print("DAMPS DIAGNOSTICS")
        print(f"{'=' * 80}")
        for k, v in diag_summary.items():
            print(f"  {k:20s}: mean={v['mean']:.4f}  std={v['std']:.4f}  "
                  f"values={[f'{x:.4f}' for x in v['values']]}")

    # ---- Comparison with the original MMHCL paper -------------------------
    print(f"\n{'=' * 80}")
    print("COMPARISON WITH PAPER (MMHCL - Amazon Clothing)")
    print(f"{'=' * 80}")
    paper_values: dict[str, float] = {
        'recall@20': 0.0881,
        'precision@20': 0.0045,
        'ndcg@20': 0.0394,
    }
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        our_mean: float = stats[metric]['mean']
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100.0
        status: str = "[OK]" if abs(diff_pct) < 10 else "[CHECK]"
        print(
            f"  {metric.upper():14s}: "
            f"Ours={our_mean:.4f} vs Paper={paper_val:.4f} ({diff_pct:+.1f}%) {status}"
        )

    # ---- Persist machine-readable summary ---------------------------------
    summary_file: str = f"../{dataset}/multi_seed_summary.json"
    summary_data: dict[str, Any] = {
        'n_runs': len(all_results),
        'model': 'DAMPS-MMHCL (Revision 9)',
        'seeds': [r['seed'] for r in all_results],
        'config': {
            'batch_size': batch_size,
            'max_epochs': epoch,
            'lr': lr,
            'embed_size': embed_size,
            'topk': topk,
            'temperature_init': temperature,
            'rebuild_R': rebuild_R,
            'use_amp': bool(use_amp),
            'damps_apc': bool(damps_apc),
            'damps_avrf': bool(damps_avrf),
            'damps_imcf': bool(damps_imcf),
            'damps_soft_routing': bool(damps_soft_routing),
            'damps_momentum': bool(damps_momentum),
            'damps_data_driven_prior': bool(damps_data_driven_prior),
            'damps_num_categories': damps_num_categories,
            'damps_warmup_epochs': damps_warmup_epochs,
            'early_stopping_patience': early_stopping_patience,
            'early_stopping_min_epochs': early_stopping_min_epochs,
            'wandb_project': wandb_project,
        },
        'statistics': {
            k: {'mean': float(v['mean']), 'std': float(v['std'])} for k, v in stats.items()
        },
        'damps_diagnostics': diag_summary,
        'individual_results': all_results,
    }

    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, indent=2)
    print(f"\n[OK] JSON summary saved to: {summary_file}")

    # ---- Persist human-readable summary -----------------------------------
    summary_txt_file: str = f"../{dataset}/multi_seed_summary.txt"
    with open(summary_txt_file, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write(f"DAMPS-MMHCL MULTI-SEED RESULTS ({len(all_results)} runs)\n")
        f.write(f"Configuration: batch_size={batch_size}, max_epochs={epoch}, "
                f"lr={lr}, rebuild_R={rebuild_R}, use_amp={bool(use_amp)}\n")
        f.write(f"DAMPS switches: APC={damps_apc} AVRF={damps_avrf} IMCF={damps_imcf} "
                f"Soft-Routing={damps_soft_routing} Momentum={damps_momentum}\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Seeds used: {seeds[:len(all_results)]}\n\n")

        for metric in metrics:
            if not stats[metric]['values']:
                continue
            f.write(f"{metric.upper()}:\n")
            f.write(f"  Mean +/- Std: {stats[metric]['mean']:.6f} +/- {stats[metric]['std']:.6f}\n")
            f.write(f"  Individual : {[f'{v:.6f}' for v in stats[metric]['values']]}\n\n")

        if diag_summary:
            f.write("\nDAMPS Diagnostics:\n")
            f.write("-" * 80 + "\n")
            for k, v in diag_summary.items():
                f.write(f"  {k:20s}: mean={v['mean']:.4f}  std={v['std']:.4f}\n")

        f.write("\nIndividual Run Results:\n")
        f.write("-" * 80 + "\n")
        for r in all_results:
            precision_str: str = (
                f"{r['precision@20']:.6f}" if r.get('precision@20') is not None else "n/a"
            )
            f.write(
                f"Seed {r['seed']:10d}: "
                f"Recall@20={r['recall@20']:.6f}, "
                f"Precision@20={precision_str}, "
                f"NDCG@20={r['ndcg@20']:.6f}\n"
            )

    print(f"[OK] Human-readable summary saved to: {summary_txt_file}")
else:
    print("\n[WARNING] No results were extracted from any DAMPS-MMHCL training run!")
    print("Please check the log files under MMHCL_DAMPS_Project/../<dataset>/ manually.")

## 5. Export Results to Excel

In [ ]:
from __future__ import annotations

from typing import Any

import pandas as pd
from openpyxl.styles import Font, PatternFill

if len(all_results) > 0:
    print("=" * 80)
    print("EXPORTING DAMPS-MMHCL RESULTS TO EXCEL")
    print("=" * 80)

    excel_file: str = f"../{dataset}/DAMPS_MMHCL_{dataset}_Results_Local.xlsx"

    # ---- Individual Results (incl. DAMPS diagnostics) -----------------------
    individual_data: list[dict[str, Any]] = []
    for i, r in enumerate(all_results, 1):
        row: dict[str, Any] = {
            'Run': i,
            'Seed': r['seed'],
            'Recall@20': r['recall@20'],
            'Precision@20': r.get('precision@20'),
            'NDCG@20': r['ndcg@20'],
            'tau_final': r.get('tau_final'),
            'tanh_sat_final': r.get('tanh_sat_final'),
            'last_nnz': r.get('last_nnz'),
            'last_avg_deg': r.get('last_avg_deg'),
        }
        individual_data.append(row)

    df_individual: pd.DataFrame = pd.DataFrame(individual_data)

    # ---- Summary statistics --------------------------------------------------
    summary_data_list: list[dict[str, Any]] = []
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        summary_data_list.append({
            'Metric': metric.upper(),
            'Mean': stats[metric]['mean'],
            'Std': stats[metric]['std'],
            'Mean +/- Std': f"{stats[metric]['mean']:.6f} +/- {stats[metric]['std']:.6f}",
        })
    df_summary: pd.DataFrame = pd.DataFrame(summary_data_list)

    # ---- Paper comparison ----------------------------------------------------
    comparison_data: list[dict[str, Any]] = []
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        our_mean: float = stats[metric]['mean']
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100.0
        comparison_data.append({
            'Metric': metric.upper(),
            'DAMPS-MMHCL (Ours)': our_mean,
            'MMHCL Paper': paper_val,
            'Difference (%)': diff_pct,
        })
    df_comparison: pd.DataFrame = pd.DataFrame(comparison_data)

    # ---- DAMPS configuration / diagnostics ----------------------------------
    config_rows: list[dict[str, Any]] = [
        {'Setting': 'Model', 'Value': 'DAMPS-MMHCL (Revision 9)'},
        {'Setting': 'Dataset', 'Value': dataset},
        {'Setting': 'Batch size', 'Value': batch_size},
        {'Setting': 'Max epochs', 'Value': epoch},
        {'Setting': 'Learning rate', 'Value': lr},
        {'Setting': 'Embedding dim', 'Value': embed_size},
        {'Setting': 'Top-k (KNN)', 'Value': topk},
        {'Setting': 'Temperature init', 'Value': temperature},
        {'Setting': 'Rebuild R (Pattern B\')', 'Value': rebuild_R},
        {'Setting': 'use_amp (bf16)', 'Value': bool(use_amp)},
        {'Setting': 'damps_apc', 'Value': bool(damps_apc)},
        {'Setting': 'damps_avrf', 'Value': bool(damps_avrf)},
        {'Setting': 'damps_imcf', 'Value': bool(damps_imcf)},
        {'Setting': 'damps_soft_routing', 'Value': bool(damps_soft_routing)},
        {'Setting': 'damps_momentum', 'Value': bool(damps_momentum)},
        {'Setting': 'damps_data_driven_prior', 'Value': bool(damps_data_driven_prior)},
        {'Setting': 'damps_num_categories', 'Value': damps_num_categories},
        {'Setting': 'damps_warmup_epochs', 'Value': damps_warmup_epochs},
        {'Setting': 'early_stopping_patience', 'Value': early_stopping_patience},
        {'Setting': 'early_stopping_min_epochs', 'Value': early_stopping_min_epochs},
        {'Setting': 'W&B project', 'Value': wandb_project},
    ]
    df_config: pd.DataFrame = pd.DataFrame(config_rows)

    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df_individual.to_excel(writer, sheet_name='Individual Results', index=False)
        df_summary.to_excel(writer, sheet_name='Summary Statistics', index=False)
        df_comparison.to_excel(writer, sheet_name='Paper Comparison', index=False)
        df_config.to_excel(writer, sheet_name='DAMPS Config', index=False)

        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            for col in ws.columns:
                max_length: int = 0
                column: str = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except Exception:
                        pass
                ws.column_dimensions[column].width = max_length + 2

            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

    print(f"\n[OK] Results exported to: {excel_file}")
    print(f"  - Sheet 1: Individual Results ({len(all_results)} runs)")
    print(f"  - Sheet 2: Summary Statistics")
    print(f"  - Sheet 3: Paper Comparison")
    print(f"  - Sheet 4: DAMPS Config")
else:
    print("No results to export.")

## 5.5 Permutation-FFT Falsifiability Ablation (Optional)

This is the falsifiability test from **Specification Section 6, Item 8** (and compliance check **INFO 4** of `DAMPS_compliance_check_report.tex`). It re-runs the full DAMPS-MMHCL pipeline twice per seed:

- **Arm A** — `--damps_permutation_fft 0` (standard 1-D rFFT, the default)
- **Arm B** — `--damps_permutation_fft 1` (input is permuted with a fixed random permutation before FFT)

After both arms have completed, a paired t-test is performed on Recall@20 and NDCG@20. The spec's fallback rule:

| `|gap|` (Recall@20) | Decision |
|---|---|
| `< 1 %` | Switch the spectral basis to **DCT-II** (the 1-D FFT axis is not informative). |
| `>= 1 %` | The standard 1-D FFT is **validated** and remains the default. |

Skip this section unless you are preparing the final paper / camera-ready ablation table — each pair adds two full training runs.

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess

# Toggle to True when you want to actually run the falsifiability test.
RUN_PERMUTATION_FFT_ABLATION: bool = False

if not RUN_PERMUTATION_FFT_ABLATION:
    print("[SKIP] Permutation-FFT ablation disabled (set RUN_PERMUTATION_FFT_ABLATION = True to run).")
else:
    # ``DAMPS_DIR`` was defined in the environment-setup cell (cell 2) and
    # is also the cwd that ``train.py`` expects (so its ``../{dataset}/...``
    # paths resolve to the workspace root).
    if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
        os.chdir(DAMPS_DIR)
    if DAMPS_DIR not in sys.path:
        sys.path.insert(0, DAMPS_DIR)

    PYTHON_EXE: str = sys.executable
    perm_seeds: list[int] = [42, 43, 44]   # 3 paired seeds is enough for INFO 4
    perm_dataset: str = 'Clothing'
    perm_epoch: int = 250

    common_flags: list[str] = [
        '--dataset', perm_dataset,
        '--epoch', str(perm_epoch),
        '--damps_apc', '1', '--damps_avrf', '1', '--damps_imcf', '1',
        '--damps_soft_routing', '1', '--damps_momentum', '1',
        '--damps_data_driven_prior', '1', '--use_amp', '1',
    ]

    for seed in perm_seeds:
        for perm_on, tag_prefix in [(0, 'perm_fft_off'), (1, 'perm_fft_on')]:
            tag: str = f"{tag_prefix}_seed{seed}"
            print("=" * 70)
            print(f"[ablation] {tag}")
            print("=" * 70)
            cmd: list[str] = [
                PYTHON_EXE, 'train.py',
                '--seed', str(seed),
                '--ablation_target', tag,
                '--damps_permutation_fft', str(perm_on),
            ] + common_flags
            env = os.environ.copy()
            env['PYTHONIOENCODING'] = 'utf-8'
            env['PYTHONUNBUFFERED'] = '1'
            r = subprocess.run(
                cmd, capture_output=True, text=True, env=env,
                encoding='utf-8', errors='replace',
            )
            if r.stdout:
                print(r.stdout[-2000:])
            if r.returncode != 0:
                print(f"[FAIL] {tag} exited with code {r.returncode}")
                if r.stderr:
                    print(r.stderr[-2000:])

    # ----- Aggregate the paired runs and run the t-test --------------------
    aggregator: str = os.path.join(DAMPS_DIR, 'scripts', '_aggregate_permutation_fft.py')
    if os.path.exists(aggregator):
        print("=" * 70)
        print("Aggregating Permutation-FFT runs and running paired t-test...")
        print("=" * 70)
        agg_cmd: list[str] = [
            PYTHON_EXE, aggregator,
            '--dataset', perm_dataset,
            '--seeds', *[str(s) for s in perm_seeds],
        ]
        env = os.environ.copy()
        env['PYTHONIOENCODING'] = 'utf-8'
        r = subprocess.run(agg_cmd, capture_output=True, text=True, env=env,
                           encoding='utf-8', errors='replace', cwd=DAMPS_DIR)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    else:
        print(f"[WARN] Aggregator script not found at {aggregator}.")

## 6. Check W&B Run Status (Optional)

In [ ]:
from __future__ import annotations

import wandb

print(f"Checking DAMPS-MMHCL runs for {wandb_entity}/{wandb_project} ...")

try:
    api: wandb.Api = wandb.Api()
    runs: wandb.apis.public.Runs = api.runs(f"{wandb_entity}/{wandb_project}")

    if len(runs) == 0:
        print("No runs found in this project yet.")
    else:
        print(f"\nFound {len(runs)} runs. Latest 5 status:")
        print("-" * 70)
        for run in runs[:5]:
            print(f"Run Name : {run.name}")
            print(f"Status   : {run.state}")
            print(f"Created  : {run.created_at}")
            # Standard MMHCL/DAMPS keys logged by ``train.py``
            for key in (
                'val/recall@20',
                'val/ndcg@20',
                'best_test_recall@20',
                'best_test_precision@20',
                'best_test_ndcg@20',
                'damps/tau',
                'damps/tanh_sat',
                'rebuild/nnz',
                'rebuild/avg_deg',
            ):
                if key in run.summary:
                    val = run.summary[key]
                    try:
                        print(f"  {key:28s}: {float(val):.4f}")
                    except (TypeError, ValueError):
                        print(f"  {key:28s}: {val}")
            print("-" * 70)
except Exception as e:
    print(f"Error fetching run status: {e}")
    print("Make sure W&B is logged in correctly.")

## Troubleshooting (DAMPS-MMHCL)

### Common Issues

1. **CUDA Out of Memory** — even with bf16 AMP enabled
   ```python
   batch_size = 512   # or 256
   knn_chunk_size = 2048  # tile the KNN distance matrix more aggressively
   ```
   You can also disable the Slim Momentum Encoder (`damps_momentum=0`) to free up to ~2 GB.

2. **`bfloat16 not supported`** — pre-Ampere GPUs cannot run the bf16 path
   ```python
   use_amp = 0   # falls back to fp32
   ```

3. **`faiss-gpu` not installed** — only required for `N >= 60_000` items
   - Default Clothing dataset has 23 033 items, so the chunked PyTorch path is used automatically.
   - Install via `pip install faiss-gpu` (Linux) or `conda install -c pytorch faiss-gpu` if you switch to a larger dataset.

4. **W&B login issues** — run `wandb login` in a terminal first

5. **Missing modules** — re-run the *Install Dependencies* cell after restarting the kernel

6. **Data not found** — ensure the multi-modal feature files exist under
   `data/Clothing/5-core/` (UI_mat_sym.pth, User_mat_rw.pth, image_feats.npy, text_feats.npy, …).

7. **Permutation-FFT ablation** — to run the falsifiability check:
   ```python
   damps_permutation_fft = 1
   ```
   The DAMPS-MMHCL spec predicts a measurable performance drop when this is enabled.

### DAMPS Diagnostic Sanity-Checks (look for these in logs)
- `tanh_sat` should stabilise in `[0.1, 0.6]` after warm-up — values near 1.0 mean AVRF logits are saturated.
- `nnz` and `avg_deg` per rebuild should plateau, not grow unboundedly (Pattern B' is working).
- `tau` (learnable InfoNCE temperature) starts at `0.1` (spec Section 3.1) and is allowed to drift; healthy runs keep it in roughly `[0.05, 0.30]` after warm-up.

### Expected Results
- **Original MMHCL (paper)** — Recall@20 = 0.0881, Precision@20 = 0.0045, NDCG@20 = 0.0394
- **DAMPS-MMHCL** — see the JSON / Excel summary produced by Step 4 / Step 5.